In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path('./analysis_data')
OUT = BASE / 'ncs'
OUT.mkdir(parents=True, exist_ok=True)

EPS = 1e-8

In [3]:
# Load required per-story metrics for both prompt sets
required_paths = {
    'coref_original_60': BASE / 'coreference' / 'coref_metrics.csv',
    'coref_54': BASE / 'coreference' / 'coref_metrics_54.csv',
    'disc_original_60': BASE / 'implicit_connectives' / 'discourse_metrics.csv',
    'disc_54': BASE / 'implicit_connectives' / 'discourse_metrics_54.csv',
    'topic_original_60': BASE / 'topic_modelling' / 'topic_switch_metrics.csv',
    'topic_54': BASE / 'topic_modelling' / 'topic_switch_metrics_54.csv',
    'mci_original_60': BASE / 'mci' / 'mci_ratio_metrics.csv',
    'mci_54': BASE / 'mci' / 'mci_ratio_metrics_54.csv',
    'char_original_60': BASE / 'character' / 'character_metrics_story_original_60.csv',
    'char_original_54': BASE / 'character' / 'character_metrics_story_original_54.csv',
    'char_large_54': BASE / 'character' / 'character_metrics_story_large_54.csv',
}

missing = [str(path) for path in required_paths.values() if not path.exists()]
if missing:
    print(missing)

df_coref_original_60 = pd.read_csv(required_paths['coref_original_60'])[['model', 'prompt', 'story_id', 'coref_ratio']].copy()
df_disc_original_60 = pd.read_csv(required_paths['disc_original_60'])[['model', 'prompt', 'story_id', 'discourse_diversity']].copy()
df_topic_original_60 = pd.read_csv(required_paths['topic_original_60'])[['model', 'prompt', 'story_id', 'topic_switch_rate']].copy()
df_mci_original_60 = pd.read_csv(required_paths['mci_original_60'])[['model', 'prompt', 'story_id', 'mci_tanh_mcc_over_gv']].copy()

df_coref_54_all = pd.read_csv(required_paths['coref_54'])[['model', 'prompt', 'story_id', 'coref_ratio']].copy()
df_disc_54_all = pd.read_csv(required_paths['disc_54'])[['model', 'prompt', 'story_id', 'discourse_diversity']].copy()
df_topic_54_all = pd.read_csv(required_paths['topic_54'])[['model', 'prompt', 'story_id', 'topic_switch_rate']].copy()
df_mci_54_all = pd.read_csv(required_paths['mci_54'])[['model', 'prompt', 'story_id', 'mci_tanh_mcc_over_gv']].copy()

for df in [
    df_coref_original_60, df_disc_original_60, df_topic_original_60, df_mci_original_60,
    df_coref_54_all, df_disc_54_all, df_topic_54_all, df_mci_54_all,
]:
    df['story_id'] = df['story_id'].astype(int)

In [5]:
# Character story-level files are split by prompt set; add prompt labels before merge
df_char_original_60 = pd.read_csv(required_paths['char_original_60'])[['model', 'story_id', 'char_coherence']].copy()
df_char_original_60['prompt'] = 'original'
df_char_original_60['story_id'] = df_char_original_60['story_id'].astype(int)

df_char_original_54 = pd.read_csv(required_paths['char_original_54'])[['model', 'story_id', 'char_coherence']].copy()
df_char_original_54['prompt'] = 'original'
df_char_original_54['story_id'] = df_char_original_54['story_id'].astype(int)

df_char_large_54 = pd.read_csv(required_paths['char_large_54'])[['model', 'story_id', 'char_coherence']].copy()
df_char_large_54['prompt'] = 'large'
df_char_large_54['story_id'] = df_char_large_54['story_id'].astype(int)

#print('character per-story rows (original_60):', len(df_char_original_60))
#print('character per-story rows (original_54):', len(df_char_original_54))
#print('character per-story rows (large_54)   :', len(df_char_large_54))
df_char_large_54.head()

,model,story_id,char_coherence,prompt
0,claude45,345,0.761594,large
1,claude45,515,0.376588,large
2,claude45,721,0.253865,large
3,claude45,723,0.462117,large
4,claude45,733,0.649322,large


In [6]:

def build_ncs_df(df_coref_in, df_disc_in, df_char_in, df_topic_in, df_mci_in):
    df_ncs_out = (
        df_coref_in
        .merge(df_disc_in, on=['model', 'prompt', 'story_id'], how='inner')
        .merge(df_char_in, on=['model', 'prompt', 'story_id'], how='inner')
        .merge(df_topic_in, on=['model', 'prompt', 'story_id'], how='inner')
        .merge(df_mci_in, on=['model', 'prompt', 'story_id'], how='inner')
    )

    components = [
        'coref_ratio',
        'discourse_diversity',
        'char_coherence',
        'topic_switch_rate',
        'mci_tanh_mcc_over_gv'
    ]

    # Arithmetic mean (raw values)
    df_ncs_out['ncs_arithmetic'] = df_ncs_out[components].mean(axis=1)

    # Geometric mean requires non-negative values
    neg_counts = (df_ncs_out[components] < 0).sum()
    print('Negative-value diagnostics (for geometric mean):')
    print(neg_counts.to_string())

    geo_components = df_ncs_out[components].clip(lower=0)
    df_ncs_out['ncs_geometric'] = np.exp(np.log(geo_components + EPS).mean(axis=1))

    return df_ncs_out

df_coref_original_54 = df_coref_54_all[df_coref_54_all['prompt'] == 'original'].copy()
df_disc_original_54 = df_disc_54_all[df_disc_54_all['prompt'] == 'original'].copy()
df_topic_original_54 = df_topic_54_all[df_topic_54_all['prompt'] == 'original'].copy()
df_mci_original_54 = df_mci_54_all[df_mci_54_all['prompt'] == 'original'].copy()

df_coref_large_54 = df_coref_54_all[df_coref_54_all['prompt'] == 'large'].copy()
df_disc_large_54 = df_disc_54_all[df_disc_54_all['prompt'] == 'large'].copy()
df_topic_large_54 = df_topic_54_all[df_topic_54_all['prompt'] == 'large'].copy()
df_mci_large_54 = df_mci_54_all[df_mci_54_all['prompt'] == 'large'].copy()

df_ncs_original_60 = build_ncs_df(
    df_coref_original_60, df_disc_original_60, df_char_original_60, df_topic_original_60, df_mci_original_60
)

df_ncs_original_54 = build_ncs_df(
    df_coref_original_54, df_disc_original_54, df_char_original_54, df_topic_original_54, df_mci_original_54
)

df_ncs_large_54 = build_ncs_df(
    df_coref_large_54, df_disc_large_54, df_char_large_54, df_topic_large_54, df_mci_large_54
)

display(df_ncs_original_60.groupby(['model', 'prompt'])['story_id'].count().reset_index(name='n_stories'))

display(df_ncs_original_54.groupby(['model', 'prompt'])['story_id'].count().reset_index(name='n_stories'))

display(df_ncs_large_54.groupby(['model', 'prompt'])['story_id'].count().reset_index(name='n_stories'))
df_ncs_large_54.head()

Negative-value diagnostics (for geometric mean):
coref_ratio             0
discourse_diversity     0
char_coherence          0
topic_switch_rate       0
mci_tanh_mcc_over_gv    3
Negative-value diagnostics (for geometric mean):
coref_ratio             0
discourse_diversity     0
char_coherence          0
topic_switch_rate       0
mci_tanh_mcc_over_gv    3
Negative-value diagnostics (for geometric mean):
coref_ratio             0
discourse_diversity     0
char_coherence          0
topic_switch_rate       0
mci_tanh_mcc_over_gv    2


,model,prompt,n_stories
0,claude45,original,60
1,gpt4o,original,60
2,human,original,60
3,internvl3,original,60
4,llama4scout,original,60
5,qwen3vl,original,60


,model,prompt,n_stories
0,claude45,original,54
1,gpt4o,original,54
2,human,original,54
3,internvl3,original,54
4,llama4scout,original,54
5,qwen3vl,original,54


,model,prompt,n_stories
0,claude45,large,54
1,gpt4o,large,54
2,human,large,54
3,internvl3,large,54
4,llama4scout,large,54
5,qwen3vl,large,54


,model,prompt,story_id,coref_ratio,discourse_diversity,char_coherence,topic_switch_rate,mci_tanh_mcc_over_gv,ncs_arithmetic,ncs_geometric
0,human,large,345,0.581147,0.244919,0.688247,0.105177,0.724346,0.468767,0.375478
1,human,large,515,0.527452,0.242543,0.478510,0.239780,0.707678,0.439193,0.401146
2,human,large,721,0.584833,0.389718,0.322683,0.448453,0.803384,0.509814,0.483771
3,human,large,723,0.924813,0.346076,0.761594,0.331438,0.630836,0.598951,0.551383
4,human,large,733,0.968704,0.371458,0.686746,0.260453,0.937305,0.644933,0.570297


In [7]:
ncs_agg_original_60 = (
    df_ncs_original_60.groupby(['model', 'prompt'])
    .agg(
        n_stories=('story_id', 'count'),
        ncs_arithmetic_mean=('ncs_arithmetic', 'mean'),
        ncs_arithmetic_std=('ncs_arithmetic', 'std'),
        ncs_geometric_mean=('ncs_geometric', 'mean'),
        ncs_geometric_std=('ncs_geometric', 'std')
    )
    .reset_index()
)

ncs_agg_original_54 = (
    df_ncs_original_54.groupby(['model', 'prompt'])
    .agg(
        n_stories=('story_id', 'count'),
        ncs_arithmetic_mean=('ncs_arithmetic', 'mean'),
        ncs_arithmetic_std=('ncs_arithmetic', 'std'),
        ncs_geometric_mean=('ncs_geometric', 'mean'),
        ncs_geometric_std=('ncs_geometric', 'std')
    )
    .reset_index()
)

ncs_agg_large_54 = (
    df_ncs_large_54.groupby(['model', 'prompt'])
    .agg(
        n_stories=('story_id', 'count'),
        ncs_arithmetic_mean=('ncs_arithmetic', 'mean'),
        ncs_arithmetic_std=('ncs_arithmetic', 'std'),
        ncs_geometric_mean=('ncs_geometric', 'mean'),
        ncs_geometric_std=('ncs_geometric', 'std')
    )
    .reset_index()
)

print('NCS aggregate (original_60):')
display(ncs_agg_original_60)

print('\nNCS aggregate (original_54):')
display(ncs_agg_original_54)

print('\nNCS aggregate (large_54):')
display(ncs_agg_large_54)

NCS aggregate (original_60):


,model,prompt,n_stories,ncs_arithmetic_mean,ncs_arithmetic_std,ncs_geometric_mean,ncs_geometric_std
0,claude45,original,60,0.412596,0.118400,0.272642,0.212226
1,gpt4o,original,60,0.454290,0.115877,0.316607,0.205041
2,human,original,60,0.499944,0.142969,0.363711,0.266128
3,internvl3,original,60,0.431963,0.125952,0.303976,0.202533
4,llama4scout,original,60,0.322128,0.099867,0.059710,0.153561
5,qwen3vl,original,60,0.447752,0.111229,0.315688,0.217815



NCS aggregate (original_54):


,model,prompt,n_stories,ncs_arithmetic_mean,ncs_arithmetic_std,ncs_geometric_mean,ncs_geometric_std
0,claude45,original,54,0.420900,0.115816,0.286969,0.209187
1,gpt4o,original,54,0.454567,0.113581,0.315585,0.200708
2,human,original,54,0.503598,0.143065,0.372263,0.262045
3,internvl3,original,54,0.434540,0.119785,0.302100,0.198024
4,llama4scout,original,54,0.320031,0.097210,0.056614,0.147814
5,qwen3vl,original,54,0.450643,0.108796,0.322552,0.212353



NCS aggregate (large_54):


,model,prompt,n_stories,ncs_arithmetic_mean,ncs_arithmetic_std,ncs_geometric_mean,ncs_geometric_std
0,claude45,large,54,0.408153,0.121356,0.261866,0.210955
1,gpt4o,large,54,0.464802,0.111590,0.333987,0.205161
2,human,large,54,0.475879,0.107490,0.373321,0.204271
3,internvl3,large,54,0.402231,0.128266,0.263904,0.197606
4,llama4scout,large,54,0.381925,0.141821,0.170443,0.237877
5,qwen3vl,large,54,0.411732,0.109839,0.270102,0.200856


In [8]:
# Save outputs
df_ncs_original_60.to_csv(OUT / 'ncs_per_story_original_60.csv', index=False)
ncs_agg_original_60.to_csv(OUT / 'ncs_aggregate_original_60.csv', index=False)

df_ncs_original_54.to_csv(OUT / 'ncs_per_story_original_54.csv', index=False)
ncs_agg_original_54.to_csv(OUT / 'ncs_aggregate_original_54.csv', index=False)

df_ncs_large_54.to_csv(OUT / 'ncs_per_story_large_54.csv', index=False)
ncs_agg_large_54.to_csv(OUT / 'ncs_aggregate_large_54.csv', index=False)


In [10]:
from scipy.stats import ttest_rel

MODELS = ['human', 'claude45', 'gpt4o', 'internvl3', 'llama4scout', 'qwen3vl']

def human_model_paired_ttests_ncs(df_subset, metric_col):
    rows = []
    df_human = df_subset[df_subset['model'] == 'human'].set_index('story_id')

    for model in MODELS:
        if model == 'human':
            continue

        df_model = df_subset[df_subset['model'] == model].set_index('story_id')
        common_ids = sorted(df_human.index.intersection(df_model.index))

        if len(common_ids) < 2:
            rows.append({
                'model': model,
                't_stat': np.nan,
                'p_value': np.nan,
                'n': len(common_ids)
            })
            continue

        paired = pd.DataFrame({
            'human': df_human.loc[common_ids, metric_col],
            'model': df_model.loc[common_ids, metric_col]
        }).dropna()

        if len(paired) < 2:
            rows.append({
                'model': model,
                't_stat': np.nan,
                'p_value': np.nan,
                'n': len(paired)
            })
            continue

        t_stat, p_value = ttest_rel(paired['human'].values, paired['model'].values)

        rows.append({
            'model': model,
            't_stat': float(t_stat),
            'p_value': float(p_value),
            'n': len(paired)
        })

    result = pd.DataFrame(rows).sort_values('model').reset_index(drop=True)
    result['t_stat'] = result['t_stat'].map(lambda x: f"{x:.3f}" if pd.notna(x) else np.nan)
    result['p_value'] = result['p_value'].map(lambda x: f"{x:.3f}" if pd.notna(x) else np.nan)
    return result.set_index('model')[['t_stat', 'p_value', 'n']]


print('Arithmetic mean')

ttest_ncs_arithmetic_original_60 = human_model_paired_ttests_ncs(
    df_ncs_original_60, metric_col='ncs_arithmetic'
)
display(ttest_ncs_arithmetic_original_60)

ttest_ncs_arithmetic_original_54 = human_model_paired_ttests_ncs(
    df_ncs_original_54, metric_col='ncs_arithmetic'
)
display(ttest_ncs_arithmetic_original_54)

ttest_ncs_arithmetic_large_54 = human_model_paired_ttests_ncs(
    df_ncs_large_54, metric_col='ncs_arithmetic'
)
display(ttest_ncs_arithmetic_large_54)

print('Geometric mean')

ttest_ncs_geometric_original_60 = human_model_paired_ttests_ncs(
    df_ncs_original_60, metric_col='ncs_geometric'
)
display(ttest_ncs_geometric_original_60)

ttest_ncs_geometric_original_54 = human_model_paired_ttests_ncs(
    df_ncs_original_54, metric_col='ncs_geometric'
)
display(ttest_ncs_geometric_original_54)

ttest_ncs_geometric_large_54 = human_model_paired_ttests_ncs(
    df_ncs_large_54, metric_col='ncs_geometric'
)
display(ttest_ncs_geometric_large_54)

ttest_ncs_arithmetic_original_60.to_csv(OUT / 'ncs_paired_ttest_arithmetic_original_60.csv')
ttest_ncs_arithmetic_original_54.to_csv(OUT / 'ncs_paired_ttest_arithmetic_original_54.csv')
ttest_ncs_arithmetic_large_54.to_csv(OUT / 'ncs_paired_ttest_arithmetic_large_54.csv')
ttest_ncs_geometric_original_60.to_csv(OUT / 'ncs_paired_ttest_geometric_original_60.csv')
ttest_ncs_geometric_original_54.to_csv(OUT / 'ncs_paired_ttest_geometric_original_54.csv')
ttest_ncs_geometric_large_54.to_csv(OUT / 'ncs_paired_ttest_geometric_large_54.csv')


Arithmetic mean


,t_stat,p_value,n
model,,,
claude45,4.679,0.000,60
gpt4o,2.581,0.012,60
internvl3,3.717,0.000,60
llama4scout,8.138,0.000,60
qwen3vl,2.769,0.007,60


,t_stat,p_value,n
model,,,
claude45,4.107,0.000,54
gpt4o,2.522,0.015,54
internvl3,3.456,0.001,54
llama4scout,8.495,0.000,54
qwen3vl,2.635,0.011,54


,t_stat,p_value,n
model,,,
claude45,6.047,0.000,54
gpt4o,1.110,0.272,54
internvl3,5.564,0.000,54
llama4scout,4.622,0.000,54
qwen3vl,7.019,0.000,54


Geometric mean


,t_stat,p_value,n
model,,,
claude45,2.835,0.006,60
gpt4o,1.771,0.082,60
internvl3,2.007,0.049,60
llama4scout,8.207,0.000,60
qwen3vl,1.588,0.118,60


,t_stat,p_value,n
model,,,
claude45,2.615,0.012,54
gpt4o,2.002,0.050,54
internvl3,2.210,0.031,54
llama4scout,8.697,0.000,54
qwen3vl,1.607,0.114,54


,t_stat,p_value,n
model,,,
claude45,4.920,0.000,54
gpt4o,2.610,0.012,54
internvl3,5.159,0.000,54
llama4scout,5.791,0.000,54
qwen3vl,5.501,0.000,54


In [11]:
# Interaction test on the same 54 stories: did the human-model gap change from short to long?
from scipy.stats import ttest_rel

def paired_gap_change_test_54(df_short_54, df_long_54, metric_col):
    rows = []

    # Restrict to human rows once for clarity.
    short_human = df_short_54[df_short_54['model'] == 'human'].set_index('story_id')
    long_human = df_long_54[df_long_54['model'] == 'human'].set_index('story_id')

    for model in MODELS:
        if model == 'human':
            continue

        short_model = df_short_54[df_short_54['model'] == model].set_index('story_id')
        long_model = df_long_54[df_long_54['model'] == model].set_index('story_id')

        common_ids = sorted(
            short_human.index
            .intersection(long_human.index)
            .intersection(short_model.index)
            .intersection(long_model.index)
        )

        if len(common_ids) < 2:
            rows.append({
                'model': model,
                'n': len(common_ids),
                'gap_short_mean': np.nan,
                'gap_long_mean': np.nan,
                'did_mean': np.nan,
                't_stat': np.nan,
                'p_value': np.nan
            })
            continue

        h_short = short_human.loc[common_ids, metric_col].astype(float).values
        m_short = short_model.loc[common_ids, metric_col].astype(float).values
        h_long = long_human.loc[common_ids, metric_col].astype(float).values
        m_long = long_model.loc[common_ids, metric_col].astype(float).values

        gap_short = h_short - m_short
        gap_long = h_long - m_long
        did = gap_short - gap_long

        t_stat, p_value = ttest_rel(gap_short, gap_long)

        rows.append({
            'model': model,
            'n': len(common_ids),
            'gap_short_mean': float(np.mean(gap_short)),
            'gap_long_mean': float(np.mean(gap_long)),
            'did_mean': float(np.mean(did)),
            't_stat': float(t_stat),
            'p_value': float(p_value)
        })

    out = pd.DataFrame(rows).sort_values('model').reset_index(drop=True)
    return out

interaction_ncs_arithmetic_54 = paired_gap_change_test_54(
    df_ncs_original_54, df_ncs_large_54, metric_col='ncs_arithmetic'
)
interaction_ncs_geometric_54 = paired_gap_change_test_54(
    df_ncs_original_54, df_ncs_large_54, metric_col='ncs_geometric'
)

print('NCS arithmetic')
display(interaction_ncs_arithmetic_54)

print('NCS geometric')
display(interaction_ncs_geometric_54)

interaction_ncs_arithmetic_54.to_csv(
    OUT / 'ncs_interaction_gap_change_arithmetic_54.csv', index=False
)
interaction_ncs_geometric_54.to_csv(
    OUT / 'ncs_interaction_gap_change_geometric_54.csv', index=False
)


NCS arithmetic


,model,n,gap_short_mean,gap_long_mean,did_mean,t_stat,p_value
0,claude45,54,0.082699,0.067725,0.014973,0.855057,0.396368
1,gpt4o,54,0.049032,0.011077,0.037955,1.823294,0.073901
2,internvl3,54,0.069059,0.073648,-0.004589,-0.235342,0.814850
3,llama4scout,54,0.183567,0.093954,0.089613,2.852063,0.006180
4,qwen3vl,54,0.052955,0.064147,-0.011192,-0.541438,0.590475


NCS geometric


,model,n,gap_short_mean,gap_long_mean,did_mean,t_stat,p_value
0,claude45,54,0.085294,0.111455,-0.026161,-0.851293,0.398437
1,gpt4o,54,0.056677,0.039335,0.017343,0.597715,0.552576
2,internvl3,54,0.070162,0.109417,-0.039255,-1.323909,0.191215
3,llama4scout,54,0.315649,0.202879,0.112770,2.497469,0.015648
4,qwen3vl,54,0.049711,0.103220,-0.053509,-1.622540,0.110622
